In [115]:
import torch
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import torch.nn as nn
import os
import torch.optim as optim
from torch.utils.data import Dataset , DataLoader
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
from PIL import Image

In [116]:
!pip install kagglehub

In [117]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("muhammadadeelkaggle/birds-dataset")

print("Path to dataset files:", path)

Path to dataset files: C:\Users\Lenovo\.cache\kagglehub\datasets\muhammadadeelkaggle\birds-dataset\versions\1


In [118]:


# Define the transformations
data_transforms = transforms.Compose([
    transforms.Resize((224, 224)), # Resize to a standard size for input to common CNNs
    transforms.ToTensor(),         # Converts image (HWC) to Tensor (CHW) and scales pixels from 0-255 to 0-1
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) # Normalize with standard ImageNet values
])

# Load the dataset
# Assumes your images are in a directory named 'data/train'
train_dataset = ImageFolder(root=r'C:\Users\Lenovo\.cache\kagglehub\datasets\muhammadadeelkaggle\birds-dataset\versions\1', transform=data_transforms)


# When you iterate through train_loader, you get Tensors ready for your CNN model.


In [119]:
train_dataset.classes

['Birds dataset.jpg']

In [120]:
data_transforms

Compose(
    Resize(size=(224, 224), interpolation=bilinear, max_size=None, antialias=True)
    ToTensor()
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
)

In [121]:
from torch.utils.data import Dataset,DataLoader

In [122]:
import cv2

In [123]:
class Parrot(Dataset):
    def __init__(self, data_dir):
        self.data = []
        self.classes = ['Amazon', 'Gray', 'Macaw', 'White']

        class_folder = [
            'amazon green parrot.jpg',
            'gray parrot.jpg',
            'macaw.jpg',
            'white parrot.jpg'
        ]

        # 🔥 ONLY FIX: go one level deeper
        data_dir = os.path.join(data_dir, 'Birds dataset.jpg')

        for idx, folder in enumerate(class_folder):
            folder_path = os.path.join(data_dir, folder)

            if not os.path.isdir(folder_path):
                print(f"❌ Not a folder: {folder_path}")
                continue

            for imgName in os.listdir(folder_path):
                if imgName.lower().endswith(('.png', '.jpg', '.jpeg', '.webp')):
                    self.data.append(
                        (os.path.join(folder_path, imgName), idx)
                    )
        
        self.transform = transforms.Compose([
            transforms.Resize((224, 224)), # Resize to a standard size for input to common CNNs
            transforms.ToTensor(),         # Converts image (HWC) to Tensor (CHW) and scales pixels from 0-255 to 0-1
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) # Normalize with standard ImageNet values
        ])

        print(f"Total images loaded: {len(self.data)}")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_path, label = self.data[idx]
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # convert numpy array (from cv2) to PIL Image so torchvision transforms work
        from PIL import Image
        img = Image.fromarray(img)

        img = self.transform(img)
        return img, label

In [124]:
!pip install opencv-python

In [125]:
!pip install cv2

ERROR: Could not find a version that satisfies the requirement cv2 (from versions: none)
ERROR: No matching distribution found for cv2


In [126]:
base_path = r'C:\Users\Lenovo\.cache\kagglehub\datasets\muhammadadeelkaggle\birds-dataset\versions\1'
dataset = Parrot(base_path)


Total images loaded: 203


In [127]:
dataset

In [128]:
import os
base_path = r'C:\Users\Lenovo\.cache\kagglehub\datasets\muhammadadeelkaggle\birds-dataset\versions\1'
print(os.listdir(base_path))


['Birds dataset.jpg']


In [129]:
class CNNModel(nn.Module):
    def __init__(self,in_channels,):
        super().__init__()

        
        self.features = nn.Sequential(
            nn.Conv2d(in_channels ,out_channels=16,kernel_size=3,stride=1,padding='same'),
            nn.ReLU(),
            
            nn.MaxPool2d(kernel_size=2,stride=2),
            nn.Conv2d(16,32,kernel_size=3,stride=1,padding='same'),
            nn.ReLU(),
            
            nn.MaxPool2d(kernel_size=2,stride=2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32*56*56,128),
            nn.ReLU(),  
            nn.Dropout(0.5),
            nn.Linear(128,64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64,32),
            nn.ReLU(),
            nn.Dropout(0.7),
            nn.Linear(32,4)  # Assuming 4 classes for the output
        )
       

    
    def forward(self, x):
        x = self.features(x)
        y = self.classifier(x)
        return y

In [130]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = CNNModel(in_channels=3).to(device)

In [131]:
epochs = 25
learning_rate = 0.001
model = CNNModel(3).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(),lr=learning_rate)


In [132]:
len(dataset)

203

In [133]:
train_dataset = int(0.8 * len(dataset))
test_dataset = len(dataset) - train_dataset


In [134]:
train_dataset

162

In [135]:
from torch.utils.data import random_split

train_dataset, test_dataset = random_split(dataset, [train_dataset, test_dataset])

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False, pin_memory=True)

In [136]:
for epoch in range(epochs):
    total_loss = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()   
    avg_loss = total_loss / len(train_loader)
    print(f'Epoch [{epoch+1}/{epochs}], Loss: {avg_loss:.4f}')

Epoch [1/25], Loss: 1.5594
Epoch [2/25], Loss: 1.3858
Epoch [3/25], Loss: 1.3920
Epoch [4/25], Loss: 1.3763
Epoch [5/25], Loss: 1.3871
Epoch [6/25], Loss: 1.3493
Epoch [7/25], Loss: 1.3067
Epoch [8/25], Loss: 1.2511
Epoch [9/25], Loss: 1.1876
Epoch [10/25], Loss: 1.0002
Epoch [11/25], Loss: 0.9830
Epoch [12/25], Loss: 0.8720
Epoch [13/25], Loss: 0.8766
Epoch [14/25], Loss: 0.7113
Epoch [15/25], Loss: 0.6266
Epoch [16/25], Loss: 0.4455
Epoch [17/25], Loss: 0.5520
Epoch [18/25], Loss: 0.4213
Epoch [19/25], Loss: 0.4041
Epoch [20/25], Loss: 0.3236
Epoch [21/25], Loss: 0.2601
Epoch [22/25], Loss: 0.2263
Epoch [23/25], Loss: 0.3563
Epoch [24/25], Loss: 0.2336
Epoch [25/25], Loss: 0.2248


In [137]:
print(images.device)
print(next(model.parameters()).device)


cuda:0
cuda:0


In [138]:
model.eval()

total = 0
correct = 0

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        _, predicted = torch.max(outputs, dim=1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = correct / total
print(f"Test Accuracy: {accuracy * 100:.2f}%")


Test Accuracy: 80.49%


In [139]:
model.eval()

total = 0
correct = 0

with torch.no_grad():
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        _, predicted = torch.max(outputs, dim=1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = correct / total
print(f"Train Accuracy: {accuracy * 100:.2f}%")


Train Accuracy: 100.00%


In [140]:
# Prediction 
image_path = r'C:\Users\Lenovo\OneDrive\Desktop\Data Structures and Algorithms\Deep Learning\Pytorch\parrot-4054102_1280.jpg'
class_names = ['Amazon', 'Gray', 'Macaw', 'White']
image = Image.open(image_path).convert('RGB')
transform = data_transforms
image = transform(image).unsqueeze(0).to(device)         
model.eval()

with torch.no_grad():
    outputs = model(image)
    _, predicted = torch.max(outputs, 1)

predicted_class = class_names[predicted.item()]
print("Predicted class:", predicted_class)


Predicted class: White
